# KernelKG-GPT — Stage 3 (train + evaluate) on Colab

Trains the **KernelGAT** verifier on the Stage 1+2 triple-pool cache and
evaluates it on the test split.

- Input: `MyDrive/kernelkg_gpt/cache/stage12_{train,dev,test}.pkl`
  (produced by the Stage 1+2 notebook).
- Output: best checkpoint → `MyDrive/kernelkg_gpt/outputs/best.pt`.

Each `(claim, triple)` becomes a node `[CLS] claim [SEP] head rel tail [SEP]`;
the model scores nodes with 21 Gaussian kernels + a sentence-level GAT and
aggregates to a True/False decision (NLL loss). This is plain
torch+transformers — **no vLLM, no version pinning**.

## Before you run
1. **Runtime → GPU** (T4 is enough; A100 faster).
2. Make sure the Stage 1+2 caches exist on Drive (run the Stage 1+2 notebook
   first). `dev` cache is used for validation / early stopping.
3. Run top-to-bottom.

## Fair-comparison tip
Set `MODEL_TYPE = "concat_baseline"` to train the supervised baseline on the
*same* triples (no kernel/GAT). Compare with `MODEL_TYPE = "kernel"` to isolate
the KernelGAT architecture's contribution.


## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Dependencies
Colab already ships torch + transformers (which include `BertModel` and `get_linear_schedule_with_warmup`). We only ensure they're importable.

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

## 3. Configuration

In [ ]:
import os, pickle, random
import numpy as np
from tqdm.auto import tqdm

# ---- Model ----
MODEL_TYPE   = "kernel"            # "kernel" | "concat_baseline"
BERT_MODEL   = "bert-base-uncased"
NUM_KERNELS  = 21
NUM_LABELS   = 2
DROPOUT      = 0.1
MAX_SEQ_LEN  = 96                  # triples are short
MAX_NODES    = 10                  # KG-GPT returns up to ~30 triples; cap to 10 nodes
TRIPLE_FORMAT = "plain"            # "plain" | "separators"

# ---- Training ----
BATCH_SIZE   = 8
EVAL_BATCH   = 16
GRAD_ACCUM   = 4
LR           = 5e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS   = 5
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 3
SEED         = 42

# ---- Paths (Google Drive) ----
WORK        = "/content/drive/MyDrive/kernelkg_gpt"
CACHE_DIR   = os.path.join(WORK, "cache")
OUTPUT_DIR  = os.path.join(WORK, "outputs"); os.makedirs(OUTPUT_DIR, exist_ok=True)
train_cache = os.path.join(CACHE_DIR, "stage12_train.pkl")
dev_cache   = os.path.join(CACHE_DIR, "stage12_dev.pkl")
test_cache  = os.path.join(CACHE_DIR, "stage12_test.pkl")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    import torch; torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

for _p in (train_cache, dev_cache, test_cache):
    print(("OK   " if os.path.exists(_p) else "MISSING"), _p)
print("model_type:", MODEL_TYPE, "| output:", OUTPUT_DIR)

## 4. Dataset (graph builder + tokenization)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup

NULL_TRIPLE_TEXT = "no_triple no_relation no_triple"

def format_triple(h, r, t, mode=TRIPLE_FORMAT):
    if mode == "separators":
        return f"[H] {h} [R] {r} [T] {t}"
    return f"{h} {r} {t}"

def build_graph(claim, triple_pool, max_nodes=MAX_NODES, mode=TRIPLE_FORMAT):
    nodes = []
    for trip in triple_pool[:max_nodes]:
        h, r, t = trip
        nodes.append({"triple_text": format_triple(h, r, t, mode), "is_null": False})
    while len(nodes) < max_nodes:
        nodes.append({"triple_text": NULL_TRIPLE_TEXT, "is_null": True})
    return nodes

def _label_to_int(lab):
    if isinstance(lab, (list, tuple)): lab = lab[0] if lab else None
    if isinstance(lab, str): return 1 if lab.strip().lower() in ("true","1","supported") else 0
    return 1 if bool(lab) else 0

class FactKGGraphDataset(Dataset):
    def __init__(self, cache_path, tokenizer, max_seq_len=MAX_SEQ_LEN,
                 max_nodes=MAX_NODES, mode=TRIPLE_FORMAT):
        with open(cache_path, "rb") as f:
            records = pickle.load(f)
        if isinstance(records, dict):
            records = [{"claim": k, **v} for k, v in records.items()]
        n = len(records)
        vocab = getattr(tokenizer, "vocab_size", 0) or 0
        id_dtype = np.uint16 if vocab and vocab <= 65535 else np.int32
        self._ids  = np.zeros((n, max_nodes, max_seq_len), dtype=id_dtype)
        self._mask = np.zeros((n, max_nodes, max_seq_len), dtype=np.uint8)
        self._seg  = np.zeros((n, max_nodes, max_seq_len), dtype=np.uint8)
        self._null = np.zeros((n, max_nodes), dtype=np.bool_)
        self._labels = np.zeros((n,), dtype=np.int64)
        self.claims, self.reasoning_types = [], []
        for i, item in enumerate(tqdm(records, desc=f"tokenize {os.path.basename(cache_path)}")):
            claim_text = item["claim"]
            for k, node in enumerate(build_graph(claim_text, item["triple_pool"], max_nodes, mode)):
                enc = tokenizer(claim_text, node["triple_text"], max_length=max_seq_len,
                                padding="max_length", truncation=True, return_token_type_ids=True)
                self._ids[i, k]  = enc["input_ids"]
                self._mask[i, k] = enc["attention_mask"]
                self._seg[i, k]  = enc["token_type_ids"]
                self._null[i, k] = node["is_null"]
            self._labels[i] = _label_to_int(item.get("label"))
            self.claims.append(claim_text)
            self.reasoning_types.append(item.get("reasoning_types", []))
    def __len__(self): return len(self.claims)
    def __getitem__(self, idx):
        return {
            "input_ids": torch.from_numpy(self._ids[idx].astype(np.int64)),
            "attention_mask": torch.from_numpy(self._mask[idx].astype(np.int64)),
            "token_type_ids": torch.from_numpy(self._seg[idx].astype(np.int64)),
            "is_null": torch.from_numpy(self._null[idx].copy()),
            "label": torch.tensor(int(self._labels[idx]), dtype=torch.long),
            "claim": self.claims[idx], "reasoning_types": self.reasoning_types[idx],
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "token_type_ids": torch.stack([b["token_type_ids"] for b in batch]),
        "is_null": torch.stack([b["is_null"] for b in batch]),
        "labels": torch.stack([b["label"] for b in batch]),
        "claims": [b["claim"] for b in batch],
        "reasoning_types": [b["reasoning_types"] for b in batch],
    }
print("Stage 3 data ready.")

## 5. Model — KernelGAT verifier (+ fair baseline)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def kernel_mus(n):
    l = [1.0]
    if n == 1: return l
    b = 2.0/(n-1); l.append(1-b/2)
    for i in range(1, n-1): l.append(l[i]-b)
    return l

def kernel_sigmas(n):
    l = [0.001]
    if n == 1: return l
    return l + [0.1]*(n-1)

class KernelKGGPT(nn.Module):
    def __init__(self, bert_model=BERT_MODEL, num_kernels=NUM_KERNELS, num_labels=NUM_LABELS,
                 max_nodes=MAX_NODES, max_seq_len=MAX_SEQ_LEN, dropout=DROPOUT):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model)
        self.H = self.bert.config.hidden_size
        self.K = num_kernels; self.dropout = nn.Dropout(dropout)
        mu = torch.FloatTensor(kernel_mus(num_kernels)).view(1,1,1,num_kernels)
        sg = torch.FloatTensor(kernel_sigmas(num_kernels)).view(1,1,1,num_kernels)
        self.register_buffer("mu", mu); self.register_buffer("sigma", sg)
        self.proj_select = nn.Linear(num_kernels, 1)
        self.proj_att = nn.Linear(num_kernels, 1)
        self.proj_inference_de = nn.Linear(self.H*2, num_labels)
        self.proj_gat = nn.Sequential(nn.Linear(self.H*2, 128), nn.ReLU(True), nn.Linear(128, 1))

    def _pool_node(self, q, d, aq, ad):
        aq = aq.unsqueeze(-1); ad = ad.unsqueeze(1).unsqueeze(-1)
        sim = torch.bmm(q, d.transpose(1,2)).unsqueeze(-1)
        pv = torch.exp(-((sim-self.mu)**2)/(self.sigma**2)/2)*ad
        ps = pv.sum(2)
        ls = torch.log(torch.clamp(ps, min=1e-10))*aq
        ls = ls.sum(1)/(aq.sum(1)+1e-10)
        return self.proj_select(ls)

    def _pool_token(self, q, d, aq, ad):
        ad = ad.unsqueeze(1).unsqueeze(-1)
        sim = torch.bmm(q, d.transpose(1,2)).unsqueeze(-1)
        pv = torch.exp(-((sim-self.mu)**2)/(self.sigma**2)/2)*ad
        ls = torch.log(torch.clamp(pv.sum(2), min=1e-10))
        ls = self.proj_att(ls).squeeze(-1)
        ls = ls.masked_fill((1-aq).bool(), -1e4)
        return F.softmax(ls, dim=1)

    def _self_attention(self, inputs, hiddens, mask_text, idx, is_null):
        B,N,L,H = hiddens.shape
        own_h = hiddens[:, idx:idx+1].expand(-1,N,-1,-1)
        own_m = mask_text[:, idx:idx+1].expand(-1,N,-1)
        own_i = inputs[:, idx:idx+1].expand(-1,N,-1)
        on = F.normalize(own_h, p=2, dim=-1); en = F.normalize(hiddens, p=2, dim=-1)
        att = self._pool_token(en.reshape(-1,L,H), on.reshape(-1,L,H),
                               mask_text.reshape(-1,L), own_m.reshape(-1,L))
        att = att.view(B,N,L,1)
        denoise = (att*hiddens).sum(dim=2)
        s = self.proj_gat(torch.cat([own_i, denoise], dim=-1))
        if is_null is not None:
            s = s.masked_fill(is_null.unsqueeze(-1), -1e4)
        w = F.softmax(s, dim=1)
        return (denoise*w).sum(dim=1)

    def forward(self, input_ids, attention_mask, token_type_ids, is_null):
        B,N,L = input_ids.shape
        out = self.bert(input_ids=input_ids.view(-1,L),
                        attention_mask=attention_mask.view(-1,L),
                        token_type_ids=token_type_ids.view(-1,L))
        hidden = self.dropout(out.last_hidden_state); pooled = out.pooler_output; H = hidden.size(-1)
        mt = attention_mask.view(-1,L).float().clone(); mt[:,0] = 0.0
        mc = (1.0 - token_type_ids.view(-1,L).float())*mt
        me = token_type_ids.view(-1,L).float()*mt
        hn = F.normalize(hidden, p=2, dim=2)
        ns = self._pool_node(hn, hn, mc, me).view(B,N,1).masked_fill(is_null.unsqueeze(-1), -1e4)
        select_prob = F.softmax(ns, dim=1)
        inputs = pooled.view(B,N,H); hiddens = hidden.view(B,N,L,H); mt3 = mt.view(B,N,L)
        de = [self._self_attention(inputs, hiddens, mt3, i, is_null) for i in range(N)]
        de = torch.stack(de, dim=1)
        feat = torch.cat([inputs, de], dim=-1)
        per_node = F.softmax(self.proj_inference_de(feat), dim=-1)
        agg = torch.clamp((select_prob*per_node).sum(dim=1), min=1e-10)
        return {"logits": torch.log(agg), "node_probs": select_prob.squeeze(-1)}

class BertConcatBaseline(nn.Module):
    def __init__(self, bert_model=BERT_MODEL, num_labels=NUM_LABELS, dropout=DROPOUT, **kw):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model)
        self.H = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.H, num_labels)
    def forward(self, input_ids, attention_mask, token_type_ids, is_null):
        B,N,L = input_ids.shape
        out = self.bert(input_ids=input_ids.view(-1,L),
                        attention_mask=attention_mask.view(-1,L),
                        token_type_ids=token_type_ids.view(-1,L))
        cls = out.pooler_output.view(B,N,self.H)
        valid = (~is_null).float().unsqueeze(-1)
        pooled = (cls*valid).sum(1)/valid.sum(1).clamp(min=1.0)
        return {"logits": F.log_softmax(self.classifier(self.dropout(pooled)), dim=-1), "node_probs": None}

def build_model():
    if MODEL_TYPE == "kernel": return KernelKGGPT()
    if MODEL_TYPE == "concat_baseline": return BertConcatBaseline()
    raise ValueError(MODEL_TYPE)

print("Stage 3 model ready.")

## 6. Train (with dev early-stopping) + per-reasoning-type eval

In [ ]:
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL)

train_ds = FactKGGraphDataset(train_cache, tokenizer)
dev_ds   = FactKGGraphDataset(dev_cache,   tokenizer)
test_ds  = FactKGGraphDataset(test_cache,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate_fn)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct=total=0; stats=defaultdict(lambda:{"c":0,"t":0})
    for batch in loader:
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                    batch["token_type_ids"].to(device), batch["is_null"].to(device))
        preds = out["logits"].argmax(-1).cpu(); labels = batch["labels"]
        correct += (preds==labels).sum().item(); total += labels.size(0)
        for i, types in enumerate(batch["reasoning_types"]):
            hit = int(preds[i].item()==labels[i].item())
            for t in types: stats[t]["t"]+=1; stats[t]["c"]+=hit
    return correct/max(total,1), stats

def train():
    set_seed()
    model = build_model().to(device)
    print("model_type:", MODEL_TYPE)
    criterion = nn.NLLLoss()
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = (len(train_loader)*NUM_EPOCHS)//max(1,GRAD_ACCUM)
    sched = get_linear_schedule_with_warmup(optim, int(total_steps*WARMUP_RATIO), total_steps)
    best, patience = 0.0, 0
    for epoch in range(NUM_EPOCHS):
        model.train(); optim.zero_grad(); running=0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch+1}")):
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                        batch["token_type_ids"].to(device), batch["is_null"].to(device))
            loss = criterion(out["logits"], batch["labels"].to(device))
            (loss/GRAD_ACCUM).backward(); running += loss.item()
            if (step+1)%GRAD_ACCUM==0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step(); sched.step(); optim.zero_grad()
        acc, stats = evaluate(model, dev_loader)
        print(f"[epoch {epoch+1}] train_loss={running/len(train_loader):.4f}  dev_acc={acc:.4f}")
        for t,s in sorted(stats.items()):
            if s["t"]: print(f"    {t:22s}: {s['c']/s['t']:.4f} ({s['t']})")
        if acc>best:
            best=acc; patience=0
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best.pt"))
            print("    * saved best")
        else:
            patience+=1
            if patience>=EARLY_STOP_PATIENCE:
                print("Early stopping."); break
    print(f"Best dev acc: {best:.4f}")
    return model

model = train()

## 7. Final evaluation on the test split

In [ ]:
# ---- Final evaluation on the test split ----
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best.pt"), map_location=device))
acc, stats = evaluate(model, test_loader)
print("="*50)
print(f"TEST accuracy: {acc:.4f}")
print("Per reasoning type:")
for t,s in sorted(stats.items()):
    if s["t"]: print(f"  {t:22s}: {s['c']/s['t']:.4f} ({s['t']})")

## 8. (Optional) Save test predictions + metrics to Drive

In [ ]:
import json
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best.pt"), map_location=device))
model.eval()

rows, correct, total = [], 0, 0
from collections import defaultdict
type_stats = defaultdict(lambda: {"c": 0, "t": 0})
with torch.no_grad():
    for batch in test_loader:
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                    batch["token_type_ids"].to(device), batch["is_null"].to(device))
        probs = out["logits"].exp().cpu()           # logits are log-probs
        preds = probs.argmax(-1)
        for i in range(len(batch["claims"])):
            p = int(preds[i]); y = int(batch["labels"][i])
            correct += int(p == y); total += 1
            for t in batch["reasoning_types"][i]:
                type_stats[t]["t"] += 1; type_stats[t]["c"] += int(p == y)
            rows.append({"claim": batch["claims"][i], "pred": p, "label": y,
                         "p_true": float(probs[i, 1]),
                         "reasoning_types": batch["reasoning_types"][i]})

metrics = {"model_type": MODEL_TYPE, "test_accuracy": correct / max(total, 1),
           "n": total,
           "per_type": {t: s["c"] / s["t"] for t, s in type_stats.items() if s["t"]}}
with open(os.path.join(OUTPUT_DIR, f"test_metrics_{MODEL_TYPE}.json"), "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, f"test_predictions_{MODEL_TYPE}.jsonl"), "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("Saved metrics + predictions to", OUTPUT_DIR)

## 9. Notes

- **Memory**: effective batch = `BATCH_SIZE × GRAD_ACCUM`. Each item is
  `MAX_NODES` BERT passes, so a batch is `BATCH_SIZE × MAX_NODES` sequences.
  If you OOM, lower `BATCH_SIZE` or `MAX_NODES`, or raise `GRAD_ACCUM`.
- **Fair baseline**: rerun with `MODEL_TYPE = "concat_baseline"`; the metrics
  file is suffixed by model type so both are kept. Report
  `kernel` vs `concat_baseline` to show the architecture's contribution.
- **Tokenization is cached in-memory** (uint16) per dataset at load time; the
  first epoch's start spends a minute tokenizing the train split.
- **Resume**: best checkpoint is saved to Drive each time dev accuracy improves.
